# Study Aberration Pairs — primary vs secondary Zernike per donut (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-05-11  
**Last Modified:** 2026-05-11  
**Status:** Draft  
**Keywords:** AOS, FAM, Danish, donut, Zernike, primary, secondary, correlation

## Description

For each per-donut FAM Zernike fit, look at the joint distribution of
a *primary* aberration term (e.g. Z5 = primary 45° astigmatism) and
its higher-radial-order *secondary* counterpart at the same azimuthal
frequency and parity (Z13 = 2nd 45° astigmatism), and ask: is the
correlation between the two aberrations the **same regardless of the
absolute level of the primary**, or does it ride along with the
primary amplitude?

Procedure per pair `(j_primary, j_secondary)`:

1. Split donuts into 4 quartiles by the primary aberration value.
2. For each quartile, 2-D histogram density of `Z_{j_sec}` vs
   `Z_{j_pri}` with an OLS line `Z_sec = m · Z_pri + b` and
   annotated slope / intercept / Pearson r / n.
3. Stack the 4 quartile panels on a single page.  If the slope is
   constant across quartiles → constant correlation.  If the slope
   varies → correlation depends on the absolute primary level.

Runs separately for chunk1 (bin_2x) and chunk2 (bin_1x) — one PDF
per binning so the two can be compared side-by-side.

All rotator angles and elevations are included (`Full Array Mode`
triplets, no geometric cut).

**Output:** PDFs + a per-pair / per-quartile summary parquet in
`output/study_aberrationpairs/`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-05-11 | Aaron Roodman | Initial version: 10 primary/secondary pairs, 4 quartiles per pair, bin_1x and bin_2x. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Per-pair Analysis](#analysis)
6. [Quartile Density Plots](#plots)
7. [Output Tables](#output)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input donut parquet files -----
# Each chunk is processed separately; the visits sidecar (same stem
# + '_visits.parquet') is used for the bad-fit / program filtering.
# Default is the chunk1 / chunk2 March 15-27 pair from runs.yaml.
chunks = {
    'bin_2x': 'output/aos_fam_danish_v1_triplets_bin_2x_20260315_20260327.parquet',
    'bin_1x': 'output/aos_fam_danish_v1_triplets_bin_1x_20260315_20260327.parquet',
}

# Coordinate system used for the per-donut zk array.
coord_sys = 'OCS'

# ----- Primary -> Secondary aberration pairs (pupil Noll j) -----
# Each entry pairs a low-radial-order Zernike with its next
# higher-radial-order partner at the same (|m|, parity).
aberration_pairs = [
    (4,  11),   # Defocus      ->  Spherical          (m=0)
    (5,  13),   # Astig45      ->  2ndAstig45         (m=2 sin)
    (6,  12),   # Astig0       ->  2ndAstig0          (m=2 cos)
    (7,  17),   # Coma_y       ->  2ndComa_y          (m=1 sin)
    (8,  16),   # Coma_x       ->  2ndComa_x          (m=1 cos)
    (9,  19),   # Trefoil_y    ->  2ndTrefoil_y       (m=3 sin)
    (10, 18),   # Trefoil_x    ->  2ndTrefoil_x       (m=3 cos)
    (11, 22),   # Spherical    ->  2ndSpherical       (m=0)
    (14, 26),   # Tetrafoil_x  ->  2ndTetrafoil_x     (m=4 cos)
    (15, 25),   # Tetrafoil_y  ->  2ndTetrafoil_y     (m=4 sin)
]

# ----- Quartile split + plot styling -----
n_quartiles            = 4
density_n_bins_per_axis = 80
# Per-axis percentile clipping inside each panel so a few outliers
# don't blow up the plotted range.
density_plo_pct = 1.0
density_phi_pct = 99.0

# Drop visits flagged as bad in the fits parquet?
drop_bad_visits = True
# Optional science_program filter (None = no constraint).
program_filter  = None
# Fit prefix (used only to read the bad_fit flag).
fit_prefix      = 'z1toz6'

# ----- Output -----
output_dir         = 'output/study_aberrationpairs'
# One PDF per chunk; output_pdf[chunk_label] is auto-built below.
output_pdf_template = f'{output_dir}/study_aberrationpairs_{{chunk}}.pdf'
output_table_path   = f'{output_dir}/study_aberrationpairs_summary.parquet'

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LogNorm
from astropy.table import QTable
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
from common.zernike_names import NOLL_NAMES

setup_plotting()

print('Ready.')

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
def visits_sidecar_path(donut_parquet_path):
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_visits.parquet')


def fits_sidecar_path(donut_parquet_path):
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_fits.parquet')


def bad_visit_set(donut_parquet_path, fit_prefix, program_filter=None):
    """Return the set of (day_obs, seq_num) flagged as bad by run_dz_fit
    (or excluded by the program filter).
    """
    bad = set()
    fp = fits_sidecar_path(donut_parquet_path)
    if not fp.exists():
        print(f'  (no fits parquet at {fp.name}; skipping bad-fit cut)')
        return bad
    ft = QTable.read(str(fp))
    # bad_fit flag
    bf_col = None
    for cand in (f'{fit_prefix}_bad_fit', 'bad_fit'):
        if cand in ft.colnames:
            bf_col = cand
            break
    if bf_col is not None:
        bad_mask = np.asarray(ft[bf_col]).astype(bool)
    else:
        bad_mask = np.zeros(len(ft), dtype=bool)
    if program_filter is not None and 'science_program' in ft.colnames:
        sp = np.asarray(ft['science_program']).astype(str)
        if isinstance(program_filter, (list, tuple, set)):
            prog_set = {str(p) for p in program_filter}
            wrong_prog = ~np.array([s in prog_set for s in sp])
        else:
            wrong_prog = (sp != str(program_filter))
        bad_mask = bad_mask | wrong_prog
    bad_dobs = np.asarray(ft['day_obs']).astype(int)
    bad_snum = np.asarray(ft['seq_num']).astype(int)
    for d, s in zip(bad_dobs[bad_mask], bad_snum[bad_mask]):
        bad.add((int(d), int(s)))
    return bad


def iZs_from_visits_sidecar(donut_parquet_path):
    """Return the canonical pupil Noll j list from the visits sidecar's
    nollIndices column (one list per visit, identical across rows).
    """
    sp = visits_sidecar_path(donut_parquet_path)
    vt = QTable.read(str(sp))
    if 'nollIndices' not in vt.colnames:
        raise ValueError(f'{sp} has no nollIndices column')
    return [int(j) for j in np.asarray(vt['nollIndices'][0]).tolist()]


def load_donut_zk(donut_parquet_path, *, fit_prefix='z1toz6',
                  coord_sys='OCS', drop_bad_visits=True,
                  program_filter=None, max_visits=None):
    """Stream a donut parquet and stack zk arrays for all matched donuts.

    Returns (zk, day_obs, seq_num, iZs) where zk has shape (N_donut, n_j).
    Memory cost ~ 21 floats × N_donut × 8 B ≈ 170 B per donut.
    """
    iZs = iZs_from_visits_sidecar(donut_parquet_path)
    bad_set = (bad_visit_set(donut_parquet_path, fit_prefix,
                              program_filter=program_filter)
                if (drop_bad_visits or program_filter is not None) else set())

    pf = pq.ParquetFile(str(donut_parquet_path))
    n_row_groups = pf.num_row_groups
    if max_visits is not None:
        n_row_groups = min(n_row_groups, max_visits)

    zk_chunks   = []
    dobs_chunks = []
    snum_chunks = []
    zk_col = f'zk_{coord_sys}'
    n_skipped = 0
    print(f'Streaming {n_row_groups} visits from '
          f'{Path(donut_parquet_path).name}...')
    for i in tqdm(range(n_row_groups), desc='visits'):
        df = pf.read_row_group(i).to_pandas()
        if len(df) == 0:
            continue
        d = int(df['day_obs'].iloc[0])
        s = int(df['seq_num'].iloc[0])
        if (d, s) in bad_set:
            n_skipped += 1
            continue
        if 'matched_intra_extra' in df.columns:
            df = df[df['matched_intra_extra'].fillna(False)]
        if len(df) == 0:
            continue
        zk = np.stack(df[zk_col].values)
        zk_chunks.append(zk)
        dobs_chunks.append(
            np.full(len(df), d, dtype=int))
        snum_chunks.append(
            np.full(len(df), s, dtype=int))
    if not zk_chunks:
        return np.empty((0, len(iZs))), np.empty(0, int), np.empty(0, int), iZs
    zk    = np.concatenate(zk_chunks, axis=0)
    dobs  = np.concatenate(dobs_chunks)
    snum  = np.concatenate(snum_chunks)
    print(f'  {len(zk):,} matched donuts loaded; '
          f'{n_skipped} visits skipped (bad / program-filtered)')
    return zk, dobs, snum, iZs


def zk_index_for(j_list, j):
    """Return the zk-array index for pupil Noll j (raise on missing)."""
    if j not in j_list:
        raise KeyError(f'Z{j} not in iZs ({j_list})')
    return j_list.index(j)


def quartile_masks(values, n_quartiles=4):
    """Equal-frequency bin masks for `values`.

    Returns (masks, edges) where `masks[q]` is the bool array for
    quartile q (0 = smallest) and `edges` is the array of bin edges
    (len n_quartiles + 1).
    """
    v = np.asarray(values, dtype=float)
    finite = np.isfinite(v)
    edges = np.nanquantile(
        v[finite],
        np.linspace(0, 1, n_quartiles + 1))
    # Force unique edges in case of degeneracies
    edges = np.unique(edges)
    if len(edges) - 1 < n_quartiles:
        print(f'  WARNING: quartile_masks collapsed {n_quartiles} -> '
              f'{len(edges) - 1} bins (degenerate edges)')
    masks = []
    for q in range(len(edges) - 1):
        lo, hi = edges[q], edges[q + 1]
        if q == len(edges) - 2:  # include right edge in last bin
            masks.append(finite & (v >= lo) & (v <= hi))
        else:
            masks.append(finite & (v >= lo) & (v <  hi))
    return masks, edges


def plot_quartile_density_page(j_pri, j_sec, x_all, y_all,
                               n_quartiles=4,
                               n_bins_per_axis=80,
                               plo_pct=1.0, phi_pct=99.0,
                               chunk_label='', ncols=2):
    """One page: 2x2 grid of 2-D density panels, one per quartile of x."""
    masks, edges = quartile_masks(x_all, n_quartiles=n_quartiles)
    nrows = (len(masks) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 10),
                              layout='constrained')
    axes = np.atleast_1d(axes).ravel()
    summary = []
    for q, m in enumerate(masks):
        ax = axes[q]
        x = x_all[m]; y = y_all[m]
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]; y = y[finite]
        if len(x) < 5:
            ax.set_visible(False)
            continue
        x_lo = float(np.nanpercentile(x, plo_pct))
        x_hi = float(np.nanpercentile(x, phi_pct))
        y_lo = float(np.nanpercentile(y, plo_pct))
        y_hi = float(np.nanpercentile(y, phi_pct))
        xp = 0.02 * max(abs(x_hi - x_lo), 1e-6)
        yp = 0.02 * max(abs(y_hi - y_lo), 1e-6)
        x_lo -= xp; x_hi += xp; y_lo -= yp; y_hi += yp

        xbins = np.linspace(x_lo, x_hi, n_bins_per_axis + 1)
        ybins = np.linspace(y_lo, y_hi, n_bins_per_axis + 1)
        counts, xe, ye = np.histogram2d(x, y, bins=[xbins, ybins])
        norm = LogNorm(vmin=1, vmax=max(counts.max(), 2))
        masked = np.ma.masked_where(counts == 0, counts)
        cmap = plt.get_cmap('viridis').copy()
        cmap.set_bad('white', alpha=0)
        pcm = ax.pcolormesh(xe, ye, masked.T, cmap=cmap, norm=norm,
                            shading='auto')
        plt.colorbar(pcm, ax=ax, shrink=0.85, label='donuts / bin')

        ax.axvline(0, color='gray', lw=0.4, alpha=0.5)
        ax.axhline(0, color='gray', lw=0.4, alpha=0.5)

        # OLS fit y = m*x + b on the clipped box
        in_box = ((x >= x_lo) & (x <= x_hi)
                  & (y >= y_lo) & (y <= y_hi))
        if int(in_box.sum()) >= 5:
            coef = np.polyfit(x[in_box], y[in_box], 1)
            m_, b_ = float(coef[0]), float(coef[1])
            xf = np.array([x_lo, x_hi])
            ax.plot(xf, m_ * xf + b_, 'r-', lw=1.2, alpha=0.95)
            r = float(np.corrcoef(x[in_box], y[in_box])[0, 1])
            label = (f'slope={m_:+.4f}\n'
                     f'intercept={b_:+.4f} μm\n'
                     f'r={r:+.4f}\n'
                     f'n={int(in_box.sum())}')
            summary.append({
                'q': q + 1, 'n': int(in_box.sum()),
                'slope': m_, 'intercept': b_, 'r': r,
                'x_lo': float(edges[q]), 'x_hi': float(edges[q + 1]),
            })
        else:
            label = f'n={int(finite.sum())}'
            summary.append({'q': q + 1, 'n': int(finite.sum()),
                            'slope': np.nan, 'intercept': np.nan,
                            'r': np.nan,
                            'x_lo': float(edges[q]),
                            'x_hi': float(edges[q + 1])})
        ax.text(0.04, 0.96, label, transform=ax.transAxes, ha='left',
                va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.25', fc='white',
                          alpha=0.85, lw=0))
        ax.set_xlim(x_lo, x_hi); ax.set_ylim(y_lo, y_hi)
        ax.set_xlabel(f'Z{j_pri} ({NOLL_NAMES.get(j_pri, "?")}) [μm]',
                      fontsize=9)
        ax.set_ylabel(f'Z{j_sec} ({NOLL_NAMES.get(j_sec, "?")}) [μm]',
                      fontsize=9)
        ax.set_title(
            f'Q{q + 1}: Z{j_pri} ∈ [{edges[q]:+.3f}, {edges[q + 1]:+.3f}] μm',
            fontsize=10)
        ax.tick_params(labelsize=8)
    for q in range(len(masks), len(axes)):
        axes[q].set_visible(False)
    fig.suptitle(
        f'Z{j_sec} ({NOLL_NAMES.get(j_sec, "?")})  vs  '
        f'Z{j_pri} ({NOLL_NAMES.get(j_pri, "?")})    {chunk_label}',
        fontsize=13)
    return fig, summary

<a id='data'></a>
## 4. Data Access

In [ ]:
donut_data = {}
for label, path in chunks.items():
    if not Path(path).exists():
        raise FileNotFoundError(path)
    print(f'\n=== {label} ===')
    zk, dobs, snum, iZs = load_donut_zk(
        path, fit_prefix=fit_prefix, coord_sys=coord_sys,
        drop_bad_visits=drop_bad_visits, program_filter=program_filter)
    donut_data[label] = {
        'zk': zk, 'day_obs': dobs, 'seq_num': snum,
        'iZs': iZs,
    }
    print(f'  iZs ({len(iZs)}): {iZs}')

# Validate the requested pairs are present in every chunk's iZs.
iZs_ref = donut_data[next(iter(donut_data))]['iZs']
missing = []
for j_pri, j_sec in aberration_pairs:
    for label, info in donut_data.items():
        if j_pri not in info['iZs'] or j_sec not in info['iZs']:
            missing.append((label, j_pri, j_sec))
if missing:
    print(f'\nWARNING: missing pupil-j values in some chunks: {missing}')
    print('  these pairs will be skipped during plotting')

<a id='analysis'></a>
## 5. Per-pair Analysis

For each chunk × pair: split donuts into `n_quartiles` (default 4)
equal-frequency bins by the **primary** aberration value, fit a line
to each quartile, and collect the slope / intercept / r / n into a
long-format DataFrame.  The same arrays drive the plots in §6.

In [ ]:
# Compute and stash the summary rows.  Plots are built directly from
# the donut zk arrays in §6, so we don't need to keep per-quartile
# arrays around.
summary_rows = []
for chunk_label, info in donut_data.items():
    zk = info['zk']
    iZs = info['iZs']
    for j_pri, j_sec in aberration_pairs:
        if j_pri not in iZs or j_sec not in iZs:
            continue
        x = zk[:, iZs.index(j_pri)]
        y = zk[:, iZs.index(j_sec)]
        masks, edges = quartile_masks(x, n_quartiles=n_quartiles)
        for q, m in enumerate(masks):
            xv = x[m]; yv = y[m]
            finite = np.isfinite(xv) & np.isfinite(yv)
            xv = xv[finite]; yv = yv[finite]
            if len(xv) < 5:
                summary_rows.append({
                    'chunk': chunk_label,
                    'j_primary': int(j_pri),
                    'j_secondary': int(j_sec),
                    'quartile': q + 1,
                    'x_lo': float(edges[q]),
                    'x_hi': float(edges[q + 1]),
                    'n': int(len(xv)),
                    'slope': np.nan, 'intercept': np.nan,
                    'r': np.nan,
                })
                continue
            coef = np.polyfit(xv, yv, 1)
            r = float(np.corrcoef(xv, yv)[0, 1])
            summary_rows.append({
                'chunk': chunk_label,
                'j_primary': int(j_pri),
                'j_secondary': int(j_sec),
                'quartile': q + 1,
                'x_lo': float(edges[q]),
                'x_hi': float(edges[q + 1]),
                'n': int(len(xv)),
                'slope':     float(coef[0]),
                'intercept': float(coef[1]),
                'r':         r,
            })
df_summary = pd.DataFrame(summary_rows)
print(f'Summary rows: {len(df_summary)}  '
      f'({len(donut_data)} chunks × {len(aberration_pairs)} pairs × '
      f'{n_quartiles} quartiles)')
with pd.option_context('display.max_rows', 80,
                       'display.float_format', '{:+.4f}'.format):
    display(df_summary.head(40))

<a id='plots'></a>
## 6. Quartile Density Plots

One PDF per chunk; one page per aberration pair; 4 panels per page
(quartiles of the primary aberration).  Each panel is a 2-D log-color
histogram of `Z_{j_sec}` vs `Z_{j_pri}` with an OLS line and
annotation.

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
first_page_done = False
for chunk_label, info in donut_data.items():
    out_pdf = output_pdf_template.format(chunk=chunk_label)
    zk = info['zk']
    iZs = info['iZs']
    n_pages = 0
    print(f'\nStreaming {out_pdf}...')
    with PdfPages(out_pdf) as pdf:
        for j_pri, j_sec in aberration_pairs:
            if j_pri not in iZs or j_sec not in iZs:
                continue
            x = zk[:, iZs.index(j_pri)]
            y = zk[:, iZs.index(j_sec)]
            fig, _ = plot_quartile_density_page(
                j_pri, j_sec, x, y,
                n_quartiles=n_quartiles,
                n_bins_per_axis=density_n_bins_per_axis,
                plo_pct=density_plo_pct,
                phi_pct=density_phi_pct,
                chunk_label=chunk_label, ncols=2)
            pdf.savefig(fig, bbox_inches='tight')
            if not first_page_done:
                plt.show()
                first_page_done = True
            plt.close(fig)
            n_pages += 1
    print(f'  wrote {n_pages} pages -> {out_pdf}')

<a id='output'></a>
## 7. Output Tables

Long-format summary parquet: one row per `(chunk, j_primary,
j_secondary, quartile)` carrying the quartile edges, donut count, and
OLS fit (slope, intercept, Pearson r).

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
df_summary.to_parquet(output_table_path)
print(f'Saved {len(df_summary)} rows -> {output_table_path}')

# Quick slope-table view: pivot by (chunk, pair) so the user can see
# whether the slope is consistent across quartiles at a glance.
pivot = (df_summary
         .assign(pair=lambda d: 'Z' + d['j_primary'].astype(str)
                                 + '→Z' + d['j_secondary'].astype(str))
         .pivot_table(index=['chunk', 'pair'],
                      columns='quartile', values='slope'))
with pd.option_context('display.float_format', '{:+.4f}'.format):
    display(pivot)